Ma adatelemző ágenst építünk: eszközkészlet, **skill** és **adatszótár**, a végén saját skill a projekthez. Az Ollama fusson (`gemma4:e4b`, 8192 tokenes kontextussal); net kell (Eurostat; tartalék van). A projekt részhez kell a `projekt-mnb` mappa. Futtasd sorban: **Shift + Enter**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import ast
import inspect
import json
import operator
import re
import shutil
import urllib.request
from datetime import date
from pathlib import Path

import pandas as pd

import kurzus
from kurzus import chat, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

A 8–9. alkalom eszközei és az ágens, változtatás nélkül:

In [ ]:
# A 8. alkalomból: szám a szövegben, egy eszközhívás végrehajtása, a kért eszközök listája,
# és két eszköz (százalékos változás, mai dátum). Csak futtasd.
def szam_benne(szam, szoveg):
    """Önálló számként szerepel-e a szám a szövegben (a 13,7-ben nincs benne a 3,7). A tizedespont is jó."""
    szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)
    szam = str(szam).replace(".", ",")
    return re.search(rf"(?<![\d,]){re.escape(szam)}(?!\d|,\d)", szoveg) is not None


def eszkoz_uzenet(hivas, eszkozok=None):
    """Egy eszközhívás végrehajtása: a modell kérése (a tool_calls egy eleme) → a válaszüzenet (role: tool)."""
    eszkozok = eszkozok or ESZKOZOK
    nev = hivas["function"]["name"]
    if nev in eszkozok:
        argumentumok = json.loads(hivas["function"]["arguments"] or "{}")   # a modell szövegként küldi
        fuggveny = eszkozok[nev]                                          # a szótárból
        eredmeny = fuggveny(**argumentumok)
    else:
        eredmeny = {"hiba": f"nincs ilyen eszköz: {nev}"}
    return {"role": "tool", "tool_call_id": hivas["id"], "content": json.dumps(eredmeny, ensure_ascii=False)}


def hivott_eszkozok(uzenetek):
    """Az üzenetlistában a modell által kért eszközhívások: (név, argumentumok) párok, sorrendben."""
    hivasok = []
    for u in uzenetek:
        for h in u.get("tool_calls") or []:
            hivasok.append((h["function"]["name"], json.loads(h["function"]["arguments"] or "{}")))
    return hivasok


def szazalekos_valtozas(regi, uj):
    """Hány százalékkal változott az érték a régihez képest, két tizedesre."""
    return round((uj - regi) / regi * 100, 2)


ESZKOZ_VALTOZAS = {
    "type": "function",
    "function": {
        "name": "szazalekos_valtozas",
        "description": "Kiszámolja, hány százalékkal nőtt (vagy csökkent) egy érték, például egy árindex "
                       "vagy egy árfolyam, a régi értékhez képest. Két tizedesre kerekít.",
        "parameters": {
            "type": "object",
            "properties": {
                "regi": {"type": "number", "description": "a kiinduló érték"},
                "uj": {"type": "number", "description": "az új érték"},
            },
            "required": ["regi", "uj"],
        },
    },
}


def mai_datum():
    """A mai dátum ÉÉÉÉ-HH-NN formában."""
    return date.today().isoformat()


ESZKOZ_DATUM = {
    "type": "function",
    "function": {
        "name": "mai_datum",
        "description": "A mai dátum, ÉÉÉÉ-HH-NN formában. Akkor használd, ha a válaszhoz tudni kell, milyen nap van ma.",
        "parameters": {"type": "object", "properties": {}},
    },
}


ESZKOZOK = {"szazalekos_valtozas": szazalekos_valtozas, "mai_datum": mai_datum}   # név → függvény

In [ ]:
# A 9. alkalom ágense: lépés, ciklus, nyomvonal. Csak futtasd.
RENDSZER = ("Magyarul válaszolj, röviden. Adatot és számolást csak az eszközökkel szerezz: számot csak az "
            "eszközök eredményéből írj. Ha egy eszköz hibát ad, olvasd el, és javítsd a hívást.")


NAPLO = []   # lépésenként: a kérés tokenszáma és a kért eszközök


def rovid(szoveg, hossz=150):
    """Hosszú eszközeredmény rövidítve, a kiíráshoz (a modell a teljeset kapja)."""
    return szoveg if len(szoveg) <= hossz else szoveg[:hossz] + f"… ({len(szoveg)} karakter)"


def lepes(uzenetek, leirasok):
    """Egy lépés: modellhívás; ha eszközt kér, lefuttatjuk, és az eredményt az előzményekbe írjuk.
    True-t ad, ha a modell szövegesen válaszolt (kész)."""
    valasz = chat(uzenetek, eszkozok=leirasok, gondolkodas="none")
    uzenetek.append(valasz)
    hivasok = valasz.get("tool_calls") or []
    NAPLO.append({"lepes": len(NAPLO) + 1, "tokenek": utolso_hasznalat.get("prompt_tokens", 0),
                  "eszkozok": [h["function"]["name"] for h in hivasok]})
    print(f"{len(NAPLO)}. lépés (a kérés {NAPLO[-1]['tokenek']} token)")
    if not hivasok:
        print("   VÁLASZ ", valasz["content"])
        return True
    for hivas in hivasok:
        uzenet = eszkoz_uzenet(hivas)
        print(f"   ESZKÖZ  {hivas['function']['name']}({hivas['function']['arguments']}) -> {rovid(uzenet['content'])}")
        uzenetek.append(uzenet)
    return False


def agens(kerdes, leirasok, max_lepes=6, rendszer=None):
    """Az ágensciklus: lépés lépés után, amíg a modell szövegesen nem válaszol, legfeljebb max_lepes lépésig."""
    uzenetek = [{"role": "system", "content": rendszer or RENDSZER}, {"role": "user", "content": kerdes}]
    NAPLO.clear()
    lepes_szam = 0
    while lepes_szam < max_lepes:
        lepes_szam += 1
        if lepes(uzenetek, leirasok):
            return uzenetek
    print(f"LÉPÉSKORLÁT: {max_lepes} lépés után sincs válasz, megállok.")
    return uzenetek


def nyomvonal(uzenetek):
    """Az ágens útja lépésenként: mit kért, és mit kapott vissza (ezt nézi a kiértékelés is)."""
    for u in uzenetek:
        if u["role"] == "assistant" and u.get("tool_calls"):
            print("KÉR    ", ", ".join(f"{h['function']['name']}({h['function']['arguments']})" for h in u["tool_calls"]))
        elif u["role"] == "tool":
            print("KAP    ", rovid(u["content"], 100))
        elif u["role"] == "assistant":
            print("VÁLASZ ", rovid(u["content"] or "", 100))


def eszkozbol_jott(szam, uzenetek, eszkoz=None):
    """Igaz, ha a szám egy eszköz eredményében szerepel (ha az eszköz neve meg van adva, annak az eredményében)."""
    hivo = {h["id"]: h["function"]["name"] for u in uzenetek for h in (u.get("tool_calls") or [])}
    return any(szam_benne(szam, u["content"]) for u in uzenetek
               if u["role"] == "tool" and (eszkoz is None or hivo.get(u["tool_call_id"]) == eszkoz))

## 1. Eszközkészlet egy gazdasági kérdéshez

Négy eszköz: `mai_datum()` (8. alkalom), `munkanelkuliseg(orszag, tol, ig)` (Eurostat `une_rt_a`, élőben; tartalék: `data/munkanelkuliseg.csv`, leírása: `data/munkanelkuliseg.md`), `fajl_olvasas(nev)` (csak a `data/` mappából), `szamol(kifejezes)` (számtani kifejezés, `eval` nélkül). Olvasd el az adatlekérőt: mit ad vissza hiba esetén?

In [ ]:
EUROSTAT_API = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
MUNKA_TARTALEK = pd.read_csv("data/munkanelkuliseg.csv", sep=";")
_letoltve = {}   # egy futáson belül egy országot csak egyszer kérünk le


def _munka_sorozat(orszag):
    """Egy ország teljes éves sora 2015-től ({év: ráta}) és a származása: élőben, ennek híján a tartalékból."""
    if orszag not in _letoltve:
        url = (f"{EUROSTAT_API}une_rt_a?geo={orszag}&age=Y15-74&sex=T&unit=PC_ACT"
               "&sinceTimePeriod=2015&format=JSON&lang=en")
        try:
            keres = urllib.request.Request(url, headers={"User-Agent": "kurzus-notebook"})
            with urllib.request.urlopen(keres, timeout=30) as valasz:
                adat = json.loads(valasz.read())
            evek = {i: int(t) for t, i in adat["dimension"]["time"]["category"]["index"].items()}
            _letoltve[orszag] = ({evek[int(n)]: v for n, v in adat["value"].items()}, "élő (Eurostat)")
        except Exception as hiba:
            print(f"(Nem érem el az Eurostatot: {type(hiba).__name__}. A mentett adatot használom: data/munkanelkuliseg.csv)")
            sorok = MUNKA_TARTALEK[MUNKA_TARTALEK["orszag"] == orszag]
            _letoltve[orszag] = (dict(zip(sorok["ev"], sorok["rata"])), "mentett tartalék (data/munkanelkuliseg.csv)")
    return _letoltve[orszag]


def munkanelkuliseg(orszag, tol, ig=None):
    """Éves munkanélküliségi ráta (%, 15-74 évesek) egy országra, évenként, az Eurostattól (une_rt_a)."""
    tol, ig = int(tol), int(ig or tol)
    sorozat, honnan = _munka_sorozat(orszag)
    if not sorozat:
        return {"hiba": f"ismeretlen ország: {orszag}; a kétbetűs Eurostat-kód kell (pl. HU), vagy EU27_2020"}
    adatok = {ev: r for ev, r in sorted(sorozat.items()) if tol <= ev <= ig}
    if not adatok:
        return {"hiba": f"nincs adat erre: {tol}–{ig}; elérhető: {min(sorozat)}–{max(sorozat)}"}
    return {"orszag": orszag, "mutato": "munkanélküliségi ráta, 15-74 évesek, a munkaerő %-ában",
            "forras": "Eurostat une_rt_a", "adat_honnan": honnan, "elerheto": f"{min(sorozat)}–{max(sorozat)}",
            "adatok": adatok}


print(munkanelkuliseg("HU", 2021, 2025))
print(munkanelkuliseg("HU", 2030, 2031))

### Feladat 1.1 – Copilottal

> Írd meg a `munkanelkuliseg(orszag, tol, ig)` függvény eszközleírását `ESZKOZ_MUNKANELKULISEG` néven, a 8. alkalom `ESZKOZ_VALTOZAS` mintájára (OpenAI-formátum: `type`, `function` → `name`, `description`, `parameters`). A leírás mondja meg, mit ad (éves munkanélküliségi ráta, %, 15–74 évesek, Eurostat), és hogy 2015-től van adat. Paraméterek: `orszag` (szöveg: kétbetűs Eurostat-kód, pl. HU, vagy EU27_2020), `tol` és `ig` (egész szám, év). Mindhárom kötelező.

Olvasd el, mielőtt **Keep**-et nyomsz: a `name` és a paraméterek nevei egyeznek a függvényéivel?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
def _jo_leiras(l):
    """Formai ellenőrzés: a név és a paraméterek egyeznek a függvényével, típusokkal, kötelezőként."""
    try:
        f = l["function"]
        p = f["parameters"]["properties"]
        return (l["type"] == "function" and f["name"] == "munkanelkuliseg" and len(f["description"].split()) >= 5
                and set(p) == set(inspect.signature(munkanelkuliseg).parameters)
                and p["orszag"]["type"] == "string" and p["tol"]["type"] == p["ig"]["type"] == "integer"
                and set(f["parameters"]["required"]) == {"orszag", "tol", "ig"})
    except (KeyError, TypeError):
        return False


ellenoriz_feltetel("ESZKOZ_MUNKANELKULISEG", _jo_leiras,
                   "név, leírás, a függvény három paramétere típussal, mind kötelező")

Az én változatom, és a másik két új eszköz:

In [ ]:
ESZKOZ_MUNKANELKULISEG = {
    "type": "function",
    "function": {
        "name": "munkanelkuliseg",
        "description": "Éves munkanélküliségi ráta (%, 15-74 évesek) egy országra, évenként, az Eurostattól. "
                       "2015-től van adat; ha a kért évekre nincs, a hibaüzenet megmondja, mi érhető el.",
        "parameters": {
            "type": "object",
            "properties": {
                "orszag": {"type": "string", "description": "kétbetűs Eurostat-kód (pl. HU, CZ, PL), vagy EU27_2020 az EU-ra"},
                "tol": {"type": "integer", "description": "az első év, pl. 2021"},
                "ig": {"type": "integer", "description": "az utolsó év, pl. 2025"},
            },
            "required": ["orszag", "tol", "ig"],
        },
    },
}

In [ ]:
ADAT_MAPPA = Path("data").resolve()


def fajl_olvasas(nev):
    """Egy szöveges fájl (.md, .txt, .csv) a data/ mappából, legfeljebb 3000 karakter. Más mappából nem olvas."""
    utvonal = (ADAT_MAPPA / nev).resolve()
    if ADAT_MAPPA not in utvonal.parents:
        return {"hiba": "csak a data/ mappából olvashatok"}
    if utvonal.suffix not in (".md", ".txt", ".csv") or not utvonal.is_file():
        van = sorted(p.name for p in ADAT_MAPPA.glob("*") if p.suffix in (".md", ".txt", ".csv"))
        return {"hiba": f"nincs ilyen szöveges fájl: {nev}; a data/ mappában ezek vannak: {van}"}
    szoveg = utvonal.read_text(encoding="utf-8")
    return szoveg[:3000] + (f"\n… (levágva, a fájl {len(szoveg)} karakter)" if len(szoveg) > 3000 else "")


ESZKOZ_FAJL = {
    "type": "function",
    "function": {
        "name": "fajl_olvasas",
        "description": "Beolvas egy szöveges fájlt a data/ mappából (pl. egy adatszótárat, .md), hogy megtudd, "
                       "mit jelent egy adat. Csak a fájl nevét add meg, mappa nélkül.",
        "parameters": {
            "type": "object",
            "properties": {"nev": {"type": "string", "description": "a fájl neve, pl. munkanelkuliseg.md"}},
            "required": ["nev"],
        },
    },
}


MUVELETEK = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
             ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def szamol(kifejezes):
    """Kiszámol egy számtani kifejezést (+ - * / ** és zárójel), például "(4.4 - 4.0) / 4.0 * 100".
    Nem eval: a kifejezést szétszedjük, és csak számot és e hat műveletet engedünk."""
    def ertek(csucs):
        if isinstance(csucs, ast.Constant) and isinstance(csucs.value, (int, float)):
            return csucs.value
        muvelet = MUVELETEK.get(type(getattr(csucs, "op", None)))
        if isinstance(csucs, ast.BinOp) and muvelet:
            return muvelet(ertek(csucs.left), ertek(csucs.right))
        if isinstance(csucs, ast.UnaryOp) and muvelet:
            return muvelet(ertek(csucs.operand))
        raise ValueError("csak számok és a + - * / ** műveletek; tizedesjel: pont")
    try:
        return {"kifejezes": kifejezes, "eredmeny": round(ertek(ast.parse(kifejezes, mode="eval").body), 4)}
    except Exception as hiba:
        return {"hiba": f"nem tudom kiszámolni: {hiba}"}


ESZKOZ_SZAMOL = {
    "type": "function",
    "function": {
        "name": "szamol",
        "description": "Kiszámol egy számtani kifejezést (+ - * / ** és zárójel, tizedespont), pl. különbséget, "
                       "átlagot vagy százalékos változást. Mindig ezzel számolj, ne fejben.",
        "parameters": {
            "type": "object",
            "properties": {"kifejezes": {"type": "string", "description": "pl. \"(4.4 - 4.0) / 4.0 * 100\""}},
            "required": ["kifejezes"],
        },
    },
}


ESZKOZOK.update({"munkanelkuliseg": munkanelkuliseg, "fajl_olvasas": fajl_olvasas, "szamol": szamol})
ELEMZO = [ESZKOZ_DATUM, ESZKOZ_MUNKANELKULISEG, ESZKOZ_FAJL, ESZKOZ_SZAMOL]

print(fajl_olvasas("munkanelkuliseg.md")[:300], "…\n")
print(fajl_olvasas("../kurzus.py"))
print(szamol("(4.4 - 4.0) / 4.0 * 100"))

### Feladat 1.2 – olvasás

A modell ezt küldi a `szamol` eszköznek: `__import__('os').system('rm -rf data')`. Mi történik? `"a"`: lefut, és törli a `data/` mappát; `"b"`: hibaüzenetet ad vissza, nem fut le semmi; `"c"`: a notebook összeomlik.

In [ ]:
tipp_1_2 = None  # "a", "b" vagy "c"

In [ ]:
eredmeny_1_2 = szamol("__import__('os').system('rm -rf data')")
print(eredmeny_1_2)
ellenoriz("tipp_1_2", vart="b" if "hiba" in eredmeny_1_2 and Path("data").exists() else "a")

Miért nem futott le? Mit csinált volna egy `eval`-os számológép? És a `fajl_olvasas("../kurzus.py")`?

## 2. Az ágens dönti el a sorrendet

### Feladat 2.1 – olvasás

Lekéri-e az ágens a mai dátumot (`mai_datum`), mielőtt eldönti, melyik az „utóbbi 5 év”? `"igen"` vagy `"nem"`?

In [ ]:
tipp_2_1 = None  # "igen" vagy "nem"

In [ ]:
KERDES = "Hogyan alakult a magyar munkanélküliség az utóbbi 5 évben? Írj rövid összefoglalót egy táblázattal."
u_elemzo = agens(KERDES, ELEMZO, max_lepes=8)

Az ellenőrzés a nyers adatból:

In [ ]:
ellenoriz("tipp_2_1", vart="igen" if "mai_datum" in [n for n, _ in hivott_eszkozok(u_elemzo)] else "nem")

hu = MUNKA_TARTALEK[MUNKA_TARTALEK["orszag"] == "HU"].set_index("ev")["rata"]
utolso_ev = int(hu.index.max())
lekert = sorted({ev for n, a in hivott_eszkozok(u_elemzo) if n == "munkanelkuliseg"
                 for ev in range(max(int(a["tol"]), int(hu.index.min())), min(int(a.get("ig") or a["tol"]), utolso_ev) + 1)})
print("Lekért évek:", lekert, "| az utolsó év, amelyre van adat:", utolso_ev)


def szamok_rendben(valasz, evek):
    """A válaszban minden lekért év értéke szerepel (tizedesponttal vagy -vesszővel)."""
    return bool(evek) and all(szam_benne(hu[ev], valasz) for ev in evek)


ellenoriz_feltetel("u_elemzo", lambda u: szamok_rendben(u[-1]["content"], lekert),
                   "a válaszban a lekért évek értékei a nyers adat szerint")
print("Benne van az utolsó év (", utolso_ev, "):", utolso_ev in lekert)

Melyik időszakot vette az ágens, és miért? Jók a számok? Jó az időszak? Milyen a táblázat formája?

## 3. Skill: hogyan dolgozzon a modell egy feladattípuson

A skill egy mappa, benne `SKILL.md`: fejléc (**név**, **leírás**), alatta a lépések és a válasz formája. A kontextusba csak a név és a leírás kerül; a testet a modell akkor tölti be, ha kell. Az `AGENTS.md` mindig ott van; a skill név szerint hívott eljárás. A Copilot a `.github/skills/<név>/SKILL.md` fájlokat keresi. A cella oda másolja a minta-skillt:

In [ ]:
SKILL_MAPPA = Path(".github/skills")     # itt keresi a skilleket a VS Code Copilot is


def skill_fejlec(fajl):
    """A SKILL.md fejléce (a --- sorok között): name és description."""
    szoveg = Path(fajl).read_text(encoding="utf-8")
    fej = re.match(r"^---\s*\n(.*?)\n---\s*\n", szoveg, re.S)
    mezok = dict(re.findall(r"^(\w[\w-]*):\s*(.+)$", fej.group(1), re.M)) if fej else {}
    return mezok, szoveg[fej.end():] if fej else szoveg


def skillek():
    """Az elérhető skillek: név → leírás. Csak ez kerül a rendszerpromptba (a skill teste nem)."""
    return {skill_fejlec(f)[0].get("name", f.parent.name): skill_fejlec(f)[0].get("description", "")
            for f in sorted(SKILL_MAPPA.glob("*/SKILL.md"))}


def skill_betoltese(nev):
    """Egy skill teljes utasítása (a SKILL.md teste), név szerint."""
    fajl = SKILL_MAPPA / nev / "SKILL.md"
    if not fajl.exists():
        return {"hiba": f"nincs ilyen skill: {nev}; van: {list(skillek())}"}
    return skill_fejlec(fajl)[1]


ESZKOZ_SKILL = {
    "type": "function",
    "function": {
        "name": "skill_betoltese",
        "description": "Betölti egy skill teljes utasítását a neve alapján. Ha egy skill illik a feladathoz, "
                       "előbb töltsd be, és kövesd az utasítását.",
        "parameters": {
            "type": "object",
            "properties": {"nev": {"type": "string", "description": "a skill neve, pl. adat-osszefoglalo"}},
            "required": ["nev"],
        },
    },
}


def skilles_rendszer():
    """A rendszerprompt kiegészítve az elérhető skillek listájával (név + leírás)."""
    lista = "\n".join(f"- {nev}: {leiras}" for nev, leiras in skillek().items())
    return (RENDSZER + "\n\nElérhető skillek. A skill nem eszköz: a nevét ne hívd meg, hanem a skill_betoltese "
            "eszközzel töltsd be, és kövesd az utasítását:\n" + lista)


# A minta-skill a data/ mappában érkezett; a .github/skills/ mappába másoljuk, mert a Copilot ott keresi.
# Ha már ott van (és esetleg átírtad), nem írjuk felül.
cel = SKILL_MAPPA / "adat-osszefoglalo"
if not cel.exists():
    shutil.copytree("data/skillek/adat-osszefoglalo", cel)
print(Path(cel / "SKILL.md").read_text(encoding="utf-8"))

### Feladat 3.1 – olvasás

Az ágens rendszerpromptjába (`skilles_rendszer()`) mi kerül a skillből? `"a"`: a teljes `SKILL.md`; `"b"`: csak a neve és a leírása, a testét az ágens a `skill_betoltese` eszközzel kéri le; `"c"`: semmi.

In [ ]:
tipp_3_1 = None  # "a", "b" vagy "c"

In [ ]:
print(skilles_rendszer())
_test = skill_betoltese("adat-osszefoglalo")
ellenoriz("tipp_3_1", vart="b" if "adat-osszefoglalo" in skilles_rendszer() and _test.strip()[:40] not in skilles_rendszer()
          else "a" if _test.strip()[:40] in skilles_rendszer() else "c")

Ugyanaz a kérdés, skillel:

In [ ]:
ESZKOZOK["skill_betoltese"] = skill_betoltese
u_skill = agens(KERDES, ELEMZO + [ESZKOZ_SKILL], max_lepes=8, rendszer=skilles_rendszer())

In [ ]:
nevek = [n for n, _ in hivott_eszkozok(u_skill)]
lekert_s = sorted({ev for n, a in hivott_eszkozok(u_skill) if n == "munkanelkuliseg"
                   for ev in range(max(int(a["tol"]), int(hu.index.min())), min(int(a.get("ig") or a["tol"]), utolso_ev) + 1)})
print("Kért eszközök:", nevek, "| lekért évek:", lekert_s)


def skill_formaja(valasz, evek):
    """A skill formai előírásai: táblázat soronként évvel és tizedesvesszős értékkel, és egy Forrás-sor."""
    sorok_jok = all(re.search(rf"\|\s*{ev}\s*\|\s*{str(hu[ev]).replace('.', ',')}\s*\|", valasz) for ev in evek)
    return bool(evek) and sorok_jok and re.search(r"Forrás:.*une_rt_a", valasz) is not None


ellenoriz_feltetel("u_skill", lambda u: "skill_betoltese" in nevek and "mai_datum" in nevek,
                   "a nyomvonalban: a skill betöltése, és a mai dátum lekérése")
ellenoriz_feltetel("u_skill", lambda u: utolso_ev in lekert_s and skill_formaja(u[-1]["content"], lekert_s),
                   f"benne van az utolsó év ({utolso_ev}); a táblázat tizedesvesszővel, a nyers adat szerint; Forrás-sor")

Mit változtatott a skill? Jó lett az időszak? Mit hívott meg a modell, ami nem eszköz, és mi történt? Most ugyanaz a skill, más adattal:

In [ ]:
u_cseh = agens("Hogyan alakult a cseh munkanélküliség 2019 és 2023 között?", ELEMZO + [ESZKOZ_SKILL],
               max_lepes=8, rendszer=skilles_rendszer())
cz = dict(zip(*[MUNKA_TARTALEK.loc[MUNKA_TARTALEK["orszag"] == "CZ", o] for o in ("ev", "rata")]))
ellenoriz_feltetel("u_cseh", lambda u: all(re.search(rf"\|\s*{ev}\s*\|\s*{str(cz[ev]).replace('.', ',')}\s*\|", u[-1]["content"])
                                           for ev in range(2019, 2024)) and "Forrás:" in u[-1]["content"],
                   "a cseh táblázat a nyers adat szerint, tizedesvesszővel, Forrás-sorral: utómunka nélkül")

## 4. Az adatszótár: mit jelent az adat

Az MNB-csomag `inflacio` függvényével, előbb adatszótár nélkül, aztán az adatszótár egy részletével a rendszerpromptban:

> Egy elemzés szerint az infláció 2023-ban 17,6 és 18,1 százalék között alakul. Igaz lett?

In [ ]:
import importlib.util

CSOMAG_MAPPA = None   # ha a cella nem találja a csomagot: a projekt-mnb mappád útvonala, pl. "../projekt-mnb"


def csomag_betoltese(mappa=None):
    """A projektcsomag adatlekeres.py-ja modulként. Ha nincs megadva mappa, a szokásos helyeken keresi:
    a notebookok mappa mellett, és a saját mappádban, a Dokumentumok, az Asztal és a Letöltések mappában."""
    otthon = ["~/projekt-mnb", "~/Documents/projekt-mnb", "~/Desktop/projekt-mnb", "~/Downloads/projekt-mnb"]
    jeloltek = [mappa] if mappa else ["../projekt-mnb", "../../projekt-mnb", "projekt-mnb", *otthon, "../csomagok/mnb"]
    for j in jeloltek:
        fajl = Path(j).expanduser() / "adatlekeres.py"
        if fajl.exists():
            spec = importlib.util.spec_from_file_location("adatlekeres", fajl)
            modul = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(modul)
            print("A csomag:", fajl.resolve().parent)
            return modul
    raise FileNotFoundError("Nem találom az MNB-csomag adatlekeres.py-ját. Írd be a CSOMAG_MAPPA-ba a projekt-mnb "
                            "mappád teljes útvonalát (VS Code: jobb klikk a mappán, Copy Path; Windowson r\"C:\\...\" "
                            "formában). Ha még nincs meg: töltsd le az mnb.zip-et a kurzusoldal Projekt lapjáról, "
                            "és csomagold ki a notebookok mappa mellé.")


adatlekeres = csomag_betoltese(CSOMAG_MAPPA)

In [ ]:
ESZKOZ_INFLACIO = {
    "type": "function",
    "function": {
        "name": "inflacio",
        "description": "Magyarország tényleges inflációja (%) a KSH (fogyasztóiár-index) vagy az Eurostat "
                       "(HICP) adataiból. Éves átlaghoz az időszak 'ÉÉÉÉ', havi adathoz 'ÉÉÉÉ-HH'.",
        "parameters": {
            "type": "object",
            "properties": {
                "forras": {"type": "string", "enum": ["KSH", "Eurostat"], "description": "az adatforrás"},
                "tol": {"type": "string", "description": "az első időszak, pl. '2023' vagy '2023-08'"},
                "ig": {"type": "string", "description": "az utolsó időszak (ha több időszak kell)"},
            },
            "required": ["forras", "tol"],
        },
    },
}
ESZKOZOK["inflacio"] = adatlekeres.inflacio

ADATSZOTAR = """Adatszótár (a projekt adatszotar.md-jéből, részlet):
- Az MNB Inflációs jelentéseinek inflációs számai (előrejelzés és tényadat) a KSH fogyasztóiár-indexére vonatkoznak: a jelentés 1-5. táblázatának sora „Fogyasztóiár-index”. Az állításaikat ezért az inflacio eszközzel, forras="KSH" adatsorral kell összevetni.
- Az Eurostat HICP-je más mutató (uniós módszertan); csak uniós összevetéshez kell.
- Ha egy állítás (jelentés, elemzés) számáról kérdeznek, az ellenőrzéshez mindig kérd le a tényadatot; ne a saját tudásodból válaszolj."""

### Feladat 4.1 – olvasás

Adatszótár nélkül lekéri-e az ágens a tényadatot (hív-e eszközt)? Írd be: `"igen"` vagy `"nem"`.

In [ ]:
tipp_4_1 = None  # "igen" vagy "nem"

In [ ]:
ALLITAS = "Egy elemzés szerint az infláció 2023-ban 17,6 és 18,1 százalék között alakul. Igaz lett?"

print("=== Adatszótár nélkül")
u_nelkul = agens(ALLITAS, [ESZKOZ_INFLACIO], max_lepes=4)
print("\n=== Adatszótárral")
u_szotar = agens(ALLITAS, [ESZKOZ_INFLACIO], max_lepes=4, rendszer=RENDSZER + "\n\n" + ADATSZOTAR)
print("\nA kért eszközök: nélküle", hivott_eszkozok(u_nelkul), "| vele", hivott_eszkozok(u_szotar))

In [ ]:
ellenoriz("tipp_4_1", vart="igen" if hivott_eszkozok(u_nelkul) else "nem")
ksh_2023 = float(pd.read_csv("data/inflacio_eves.csv", sep=";").set_index("ev").loc[2023, "ksh_fogyasztoiar"])
ellenoriz_feltetel("u_szotar", lambda u: any(n == "inflacio" and a.get("forras") == "KSH" for n, a in hivott_eszkozok(u))
                                          and eszkozbol_jott(ksh_2023, u, "inflacio"),
                   f"adatszótárral: KSH-adat, és a {ksh_2023} az eszközből jött")

Mit változtatott az adatszótár? Mit nem? Mi az, ami biztosan eldönti, jó-e a válasz?

## 6. A projektben: saját skill a Copilotnak

A mérföldkő: **adatszótár és saját skill a projekthez**, a projekted repójában, a Copilottal (párban: közösen):

1. **Az adatszótár:** töltsd ki a csomag `adatszotar.md`-jének `TODO`-it, forrással (oldalszámmal).
2. **A skill:** a Copilot chatben `/create-skill`, és írd le, mit csináljon. Minta:

   > Hozz létre egy skillt a `.github/skills/` mappában, `mnb-osszevetes` néven. Akkor kell, ha egy MNB-jelentés előrejelzését kell összevetni a tényadattal. Lépései: a RAG-láncunkkal megkeresi az állítást (jelentés, nyomtatott oldal, idézet); az `adatlekeres.inflacio` függvénnyel lekéri a tényadatot, az `adatszotar.md` szerinti adatsorral; az összevetést kód végzi; a végén futtatja a `python ellenorzes.py valasz.md` parancsot. A válasz formája: egy táblázat (állítás, előrejelzés, tényadat, helyzet) és egy sor arról, hogy élő vagy mentett adat volt.

   Ha nincs `/create-skill`: **Configure Chat** (fogaskerék) → **Skills** → **New Skill (Workspace)**.
3. **Olvasd el**, mielőtt **Keep**: a `name` kisbetű-szám-kötőjel, és egyezik a mappa nevével? A `description` megmondja, mit csinál és **mikor** kell? Nem ismétli az `AGENTS.md`-t?
4. **Próbáld ki** új chatben: `/mnb-osszevetes A 2023. szeptemberi jelentés mekkora inflációt várt 2023-ra, és mennyi lett?`

Írd be a `SKILL.md` útvonalát, a cella formailag ellenőrzi:

In [ ]:
def skill_rendben(fajl):
    """Egy SKILL.md formai ellenőrzése a VS Code dokumentációja szerint. Az eltérések listáját adja (üres: rendben)."""
    fajl = Path(fajl)
    if not fajl.is_file():
        return [f"nincs ilyen fájl: {fajl}"]
    mezok, test = skill_fejlec(fajl)
    gondok = []
    nev, leiras = mezok.get("name", ""), mezok.get("description", "")
    if not re.fullmatch(r"[a-z0-9-]{1,64}", nev):
        gondok.append("a name csak kisbetű, szám és kötőjel lehet, legfeljebb 64 karakter")
    if nev != fajl.parent.name:
        gondok.append(f"a name ({nev!r}) egyezzen a mappa nevével ({fajl.parent.name!r})")
    if not (len(leiras.split()) >= 10 and len(leiras) <= 1024):
        gondok.append("a description mondja meg, mit csinál és mikor kell használni (legalább 10 szó, legfeljebb 1024 karakter)")
    if len(test.split()) < 40:
        gondok.append("a skill teste túl rövid: írd le a lépéseket és a válasz formáját")
    return gondok

In [ ]:
sajat_skill = None  # a SKILL.md útvonala, pl. "../projekt-mnb/.github/skills/mnb-osszevetes/SKILL.md"

In [ ]:
if sajat_skill:
    for gond in skill_rendben(sajat_skill):
        print("JAVÍTANDÓ:", gond)
ellenoriz_feltetel("sajat_skill", lambda p: skill_rendben(p) == [],
                   "a SKILL.md formailag rendben: név = mappa, leírás (mit, mikor), lépések")

Mit nem mond meg a formai ellenőrzés?

### Otthonra

A projekted repójában (párban: közösen):

1. az `adatszotar.md` `TODO`-i kitöltve, forrással (dokumentum, oldal);
2. legalább egy saját skill a `.github/skills/` mappában; a fenti cella `RENDBEN`-t ír rá;
3. **frissebb adattal újrafuttatás:** futtasd a skillt egyszer net nélkül (a csomag a `data/tartalek.csv`-ből dolgozik, és kiírja) és egyszer élő adattal. Ugyanabban a formában jött a válasz, utómunka nélkül? Ha nem, javítsd a skillt (nem a választ), és írd le a `feltetelezesek.md`-be, mit változtattál;
4. nézd át az `AGENTS.md`-t: ami csak egy feladathoz kell, az kerüljön a skillbe; ami mindig érvényes, maradjon ott.